In [ ]:
import torchvision
import torch
from PIL import Image

# Define image size 
size = (128, 128)

# Preprocess images: resize and convert to Tensor
transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(size),  # Resize to 128x128
    torchvision.transforms.ToTensor(),    # Convert to Tensor and scale pixels to [0, 1]
])

# Load training and testing datasets
train_dataset = list(torchvision.datasets.Flowers102(
    "./flowers", "train", transform=transform, download=False
))

test_dataset = list(torchvision.datasets.Flowers102(
    "./flowers", "test", transform=transform, download=False
))


# Convert a PyTorch Tensor into a PIL Image for visualization
def visualize_image(img: torch.Tensor) -> Image.Image:
    img = img.permute(1, 2, 0)  # Convert CHW to HWC
    img = (img * 255).to(torch.uint8)  # Scale pixels to [0, 255]
    return Image.fromarray(img.numpy())  # Convert to PIL Image


# Visualize the first training image
visualize_image(train_dataset[1][0])  # [0] selects the image, [1] is the label

In [ ]:
print(len(train_dataset), len(test_dataset))  # Check the number of samples in each dataset
print(train_dataset[0])

In [ ]:
import matplotlib.pyplot as plt
f, ax = plt.subplots(2,10)
for i, (img, label) in enumerate(list(train_dataset)[:20]):
    ax[i//10, i%10].imshow(visualize_image(img))
    ax[i//10, i%10].set_title(label)

class_01 = list(train_dataset)[:20]

# Nearest Neighbor Classifier

f: R --> {0,1}

- Input x
- Find closest data point (X mean, l) thuộc D in dataset D
- Return its label l

In [ ]:
def nn_classifier(x):
    dist = [((x - im).pow(2).sum().sqrt(),l) for im, l in class_01]
    return min(dist)[1]

nn_classifier(test_dataset[0][0])
accuracy = sum(nn_classifier(x) == l for x, l in list(test_dataset)[:20]) / 20
print(accuracy)

In [ ]:
[l for _, l in list(test_dataset)[:20]]

# K Nearest Neighbor Classifier
f: R --> {0,1}

- Input x
- Find k closest data point in dataset D
- Return its label l


In [ ]:
def knn_classifier(x, k = 3):
    dist = [((x - im).pow(2).sum().sqrt(),l) for im, l in class_01]
    k_closest = [l for _, l in sorted(dist)[:k]]
    return sorted(k_closest)[k//2]  # Return the median label

knn_classifier(test_dataset[0][0])
accuracy = sum(knn_classifier(x) == l for x, l in list(test_dataset)[:20]) / 20
print(accuracy)

# K Nearest Neighbor Regression
f: R --> R

- Input x
- Find K closest data points in dataset D
- Return average value


In [ ]:
def knn_classifier(x, k = 3):
    dist = [((x - im).pow(2).sum().sqrt(),l) for im, l in class_01]
    k_closest = [l for _, l in sorted(dist)[:k]]
    return torch.mean(torch.tensor(k_closest).float())

print(knn_classifier(test_dataset[0][0]))